In [ ]:
import pandas as pd
from IPython.display import display

from study.data import cohort, features, observation_batches, prediction_matrix, prediction_metadata
from study.features import build_features
from study.io import WORK, load, reset_directory, write_table
from study.models import predict
from study.observations import prepare_restrictions, restricted_inputs, restriction_settings, subset_inputs
from study.statistics import paired_difference, prediction_rows

In [ ]:
RULES = {'protected_variables': ['Height', 'MechVent']}
CONDITIONS = ['retain_0.5', 'retain_0.25']

In [ ]:
source_fits = load(WORK / 'models/source.pkl')
reset_directory(WORK / 'predictions/quantity')

In [ ]:
# Rebuild each observation draw once for all tiers
for source, target in [
    ('carevue', 'metavision'),
    ('metavision', 'mimiciv'),
    ('metavision', 'eicu')
]:
    metadata = cohort(target).sort_index()
    ids = metadata.index
    original = features(target).loc[ids]
    supports = ('shared',) if target == 'eicu' else ('original',) if source == 'carevue' else ('original', 'shared')
    protected = [column for column in original if column.split('_')[0] in RULES['protected_variables']]

    # Prepare each raw batch once for all restriction draws
    for batch, inputs in enumerate(observation_batches(target)):
        inputs = subset_inputs(inputs, ids)
        if inputs['stays'].empty:
            continue
        prepared = prepare_restrictions(inputs, ['thinning'])

        # Rebuild restricted records using shared observation draws
        for condition in CONDITIONS:
            for draw, seed in enumerate(range(10)):
                settings = restriction_settings(condition, seed)
                rebuilt = build_features(restricted_inputs(prepared, settings, RULES))
                rebuilt.loc[:, protected] = original.loc[rebuilt.index, protected]
                evaluated = prediction_metadata(metadata.loc[rebuilt.index])
                
                # Score each tier with its frozen source model
                rows = []
                for support in supports:
                    for tier in ('R1', 'R2', 'R3', 'R4'):
                        fitted = source_fits['models'][source, 0, support, tier]
                        risk = predict(fitted['model'], rebuilt)
                        rows.append(
                            prediction_rows(
                                evaluated, risk, domain = target, source = source, support = support,
                                tier = tier, repeat = 0, restriction = condition,
                            ).assign(draw = draw)
                        )

                write_table(
                    WORK / 'predictions/quantity' / f'{source}_{target}_{condition}_{draw}_{batch}.parquet',
                    pd.concat(rows, ignore_index = True),
                )

        print(source, target, 'batch', batch + 1, 'complete')

In [ ]:
# Compare paired losses under observation restrictions
rows = []
for source, domain in [
    ('carevue', 'metavision'),
    ('metavision', 'mimiciv'),
    ('metavision', 'eicu')
]:
    metadata = cohort(domain).sort_index()
    supports = ('shared',) if domain == 'eicu' else ('original',) if source == 'carevue' else ('original', 'shared')

    for support in supports:
        for tier in ('R1', 'R2', 'R3', 'R4'):
            filters = [
                ('source', '==', source),
                ('domain', '==', domain),
                ('support', '==', support),
                ('tier', '==', tier)
            ]

            # Match full input and restricted predictions on the same patients
            baseline = prediction_matrix('full', metadata.index, filters).dropna(how = 'all')
            ids = (
                baseline.index[metadata.loc[baseline.index, 'partition'].eq('test')]
                if domain == 'eicu'
                else baseline.index
            )

            paired_metadata = prediction_metadata(metadata.loc[ids])

            for restriction in CONDITIONS:
                candidate = prediction_matrix('quantity', ids, filters + [('restriction', '==', restriction)])

                # Compare restriction losses with patient and hospital weighting
                for equal_hospital in [False, True] if domain == 'eicu' else [False]:
                    rows.append(
                        dict(
                            source = source,
                            domain = domain,
                            support = support,
                            tier = tier,
                            restriction = restriction,
                            **paired_difference(
                                paired_metadata,
                                candidate,
                                baseline.loc[ids],
                                cluster = domain == 'eicu',
                                equal_hospital = equal_hospital,
                            ),
                        )
                    )

results = pd.DataFrame(rows)
write_table(WORK / 'results/quantity.parquet', results)

display(results.round(5))